In [ ]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig, AutoTokenizer
from huggingface_hub import hf_hub_download, notebook_login
import torch
from safetensors.torch import load_file
import torch.nn as nn
from functools import partial
import numpy as np
from IPython.display import display, HTML
import textwrap

In [ ]:
notebook_login()

In [ ]:
# Config
gemma_version = "google/gemma-3-4b-it"
corresponding_sae_folder = "google/gemma-scope-2-4b-pt"

# SAE config
LAYER = 22
WIDTH = "262k"
L0 = "medium"

input_prompt = "Once there was a mountain called Peak 15. Nothing was known about it. But in 1852 the surveyors found it was the highest in the world and they named it Everest"

In [ ]:
torch.set_grad_enabled(False) # avoid blowing up mem

model = AutoModelForCausalLM.from_pretrained(
    gemma_version,
    device_map='auto',
)
tokenizer =  AutoTokenizer.from_pretrained(gemma_version)

In [ ]:
path_to_params = hf_hub_download(
    repo_id=corresponding_sae_folder,
    filename=f"resid_post/layer_{LAYER}_width_{WIDTH}_l0_{L0}/params.safetensors",
)

params = load_file(path_to_params)

## Sparse AutoEncoder

In [ ]:
class JumpReLUSAE(nn.Module):
  def __init__(self, d_in, d_sae, affine_skip_connection=False):
    # Note that we initialise these to zeros because we're loading in pre-trained weights.
    # If you want to train your own SAEs then we recommend using blah
    super().__init__()
    self.w_enc = nn.Parameter(torch.zeros(d_in, d_sae))
    self.w_dec = nn.Parameter(torch.zeros(d_sae, d_in))
    self.threshold = nn.Parameter(torch.zeros(d_sae))
    self.b_enc = nn.Parameter(torch.zeros(d_sae))
    self.b_dec = nn.Parameter(torch.zeros(d_in))
    if affine_skip_connection:
      self.affine_skip_connection = nn.Parameter(torch.zeros(d_in, d_in))
    else:
      self.affine_skip_connection = None

  def encode(self, input_acts):
    pre_acts = input_acts @ self.w_enc + self.b_enc
    mask = (pre_acts > self.threshold)
    acts = mask * torch.nn.functional.relu(pre_acts)
    return acts

  def decode(self, acts):
    return acts @ self.w_dec + self.b_dec

  def forward(self, x):
    acts = self.encode(x)
    recon = self.decode(acts)
    if self.affine_skip_connection is not None:
      return recon + x @ self.affine_skip_connection
    return recon

In [ ]:
d_model, d_sae = params["w_enc"].shape
sae = JumpReLUSAE(d_model, d_sae)
sae.load_state_dict(params)
sae.cuda()

In [ ]:
def gather_acts_hook(mod, inputs, outputs, cache: dict, key: str, use_input: bool):
  """Generic hook function whic stores activations (either input or output of a particular PyTorch module)."""
  acts = inputs[0].squeeze(0) if use_input else outputs[0]  # inputs usually have a batch dim
  cache[key] = acts
  return outputs


def gather_residual_activations(model, target_layer, inputs):

  cache = {}

  handle = model.model.language_model.layers[target_layer].register_forward_hook(
        partial(gather_acts_hook, cache=cache, key="resid_post", use_input=False)
  )

  try:
    _ = model.forward(inputs)
  finally:
    handle.remove()

  return cache["resid_post"]

In [ ]:
# Prepare input
input_prompt_tokenized = tokenizer.encode(input_prompt, return_tensors="pt", add_special_tokens=True).to("cuda")

In [ ]:
target_act = gather_residual_activations(model, LAYER, input_prompt_tokenized)

sae_acts = sae.encode(target_act.to(torch.float32))
recon = sae.decode(sae_acts)

In [ ]:
reconstruction_mse = torch.mean((recon[:, 1:] - target_act[:, 1:].float()) ** 2)
target_variance = target_act[:, 1:].float().var()

fvu = reconstruction_mse / target_variance
print(f"Fraction of variance unexplained: {fvu:.2%}")

In [ ]:
l0_per_token = (sae_acts > 1).sum(-1)
print(l0_per_token.tolist())

print(f"Average L0: {l0_per_token[1:].float().mean():.2f}")

In [ ]:
def fwd_pass_with_sae_intervention(model, sae, target_layer, inputs):
  # Forward pass to get logits & hidden activations
  model_output_clean = model.forward(inputs, output_hidden_states=True)
  logits_clean = model_output_clean.logits[0]  # (seq, d_vocab)
  input_acts = model_output_clean.hidden_states[target_layer + 1][0]  # (seq, d_model)

  # Get the SAE reconstruction
  recon = sae.forward(input_acts.to(torch.float32))

  def intervene_on_target_act_hook(mod, inputs, outputs):
    outputs[0, 1:] = recon[1:]
    return outputs

  handle = model.model.language_model.layers[target_layer].register_forward_hook(intervene_on_target_act_hook)
  try:
    model_output = model.forward(inputs)
  finally:
    handle.remove()

  # Get logits from this corrupted forward pass
  logits = model_output.logits[0]

  return logits_clean, logits


def cross_entropy_loss(logits: torch.Tensor, tokens: torch.Tensor) -> torch.Tensor:
  """Measures avg cross entropy loss."""
  logprobs = logits[:-1].log_softmax(dim=-1)
  tokens = tokens[1:]
  correct_logprobs = logprobs[torch.arange(len(tokens)), tokens]
  return -correct_logprobs

In [ ]:
logits_clean, logits_sae = fwd_pass_with_sae_intervention(model, sae, LAYER, input_prompt_tokenized)
loss_clean = cross_entropy_loss(logits_clean, input_prompt_tokenized[0])
loss_sae = cross_entropy_loss(logits_sae, input_prompt_tokenized[0])

print(f"Loss (clean): {loss_clean.mean():.4f}")
print(f"Loss (corrupted): {loss_sae.mean():.4f}")
print(f"Delta loss: {loss_sae.mean() - loss_clean.mean():.4f}")

In [ ]:
top_activations, top_features = sae_acts.max(-1)
top_acts, top_latents = sae_acts.squeeze().mean(0).topk(5)

for act, idx in zip(top_acts, top_latents):
  print(f"{act:>6.1f} | {idx}")

In [ ]:
feature_idx = 831

str_toks = tokenizer.tokenize(input_prompt, add_special_tokens=True)
activations = sae_acts[ :, feature_idx].tolist()

def html_activations(str_toks: list[str], activations: list[float]):
  return "".join(
      f'<span style="background-color: rgba(255,0,0,{v}); padding: 4px 0px;">{t}</span>'
      for t, v in zip(str_toks, np.array(activations) / (1e-6 + np.max(activations)), strict=True)
  )

display(HTML(html_activations(str_toks, activations)))

In [ ]:
def generate_with_steering(model, sae, inputs, target_layer, feature_idx: int, coeff: float):

  def steering_hook(mod, inputs, outputs):
    output = outputs
    # We have to be careful about KV caching! This logic handles different cases depending on
    # whether this is the first forward pass or a cached pass.
    if output.shape[1] == 1:
      avg_norm = torch.norm(output, dim=-1)
      output += coeff * avg_norm * sae.w_dec[feature_idx]
    else:
      # avg_norm = torch.norm(output[0, 1:], dim=-1, keepdim=True)
      # output[0, 1:] += coeff * avg_norm * sae.w_dec[feature_idx]
      avg_norm = torch.norm(output[0, -1:], dim=-1, keepdim=True)
      output[0, -1:] += coeff * avg_norm * sae.w_dec[feature_idx]

    return outputs

  handle = model.model.language_model.layers[target_layer].register_forward_hook(steering_hook)
  try:
    outputs = model.generate(input_ids=inputs, max_new_tokens=80, do_sample=False)
    output_str = tokenizer.decode(outputs[0])
  finally:
    handle.remove()

  return output_str.split("<start_of_turn>model")[1].strip()

def format_prompt(user_prompt: str) -> str:
  return f"""<start_of_turn>user
{user_prompt}<end_of_turn>
<start_of_turn>model
"""

user_prompt = "The chickens are my favourite animals."
inputs = tokenizer.encode(format_prompt(user_prompt), return_tensors="pt", add_special_tokens=True).to("cuda")

print(user_prompt)
print("======================= NO STEERING =======================")
output_str = generate_with_steering(
    model=model,
    sae=sae,
    inputs=inputs,
    target_layer=LAYER - 8,
    feature_idx=feature_idx,
    coeff=0.0,
)
print(textwrap.fill(output_str))
print("======================= STEERING =======================")
output_str_steered = generate_with_steering(
    model=model,
    sae=sae,
    inputs=inputs,
    target_layer=LAYER - 8,
    feature_idx=feature_idx,
    coeff=0.07,
)
print(textwrap.fill(output_str_steered))